# LaBSE Indic Alignment Fine-tuning Notebook

This notebook fine-tunes `sentence-transformers/LaBSE` for Indic alignment using contrastive learning.

**This edited version trains for 10 epochs and includes checkpointing.**

**Important split rule**

- Use **Samanantar / BPCC / another parallel corpus** for training.
- Keep **IN22-Gen / IN22-Conv** only for evaluation.
- Do not judge success only by mean gold-pair cosine. Re-run your benchmark after fine-tuning and compare cosine gap, specificity, balanced accuracy, ROC-AUC, and retrieval metrics.

The dataset-loading section is intentionally left blank so you can insert your own Samanantar/BPCC loading logic.

**Checkpointing behavior**

- Saves rolling checkpoints during training.
- Saves the best validation model separately.
- Saves the final model separately.
- Saves the exact train/validation split used, so you can reproduce the run.
- Can resume from the latest checkpoint after a runtime disconnect.


## 1. Install dependencies

Run this cell first.  
In Colab, after installing, it is safer to do: **Runtime → Restart runtime**, then continue from the imports cell.

In [ ]:
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    _guard_dir = _candidate / "scripts"
    if (_guard_dir / "import_guard.py").exists():
        if str(_guard_dir) not in sys.path:
            sys.path.insert(0, str(_guard_dir))
        break
else:
    raise RuntimeError("Could not locate scripts/import_guard.py. Run this notebook from the WSAI workspace or copy the guard module alongside it.")

from import_guard import install_pandas_guards
install_pandas_guards()


In [ ]:
!pip -q install -U sentence-transformers datasets accelerate transformers huggingface_hub pandas numpy scikit-learn tqdm

## 2. Imports, seed, runtime detection

In [ ]:
import os
import random
import math
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import TranslationEvaluator

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IS_COLAB = "COLAB_GPU" in os.environ or "google.colab" in str(get_ipython())
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

print("Device:", DEVICE)
print("Running in Colab:", IS_COLAB)
print("Running in Kaggle:", IS_KAGGLE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("GPU memory GB:", round(total_gb, 2))


## 3. Optional: Mount Google Drive / set output folders

For Kaggle, this will use `/kaggle/working`.  
For Colab, it will try to mount Google Drive. You can also skip Drive and use `/content`.

In [ ]:
USE_GOOGLE_DRIVE = IS_COLAB

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE_DIR = Path("/content/drive/MyDrive/labse_indic_finetuning")
    except Exception as e:
        print("Drive mount failed. Falling back to /content.")
        print("Error:", e)
        BASE_DIR = Path("/content/labse_indic_finetuning")
elif IS_KAGGLE:
    BASE_DIR = Path("/kaggle/working/labse_indic_finetuning")
else:
    BASE_DIR = Path("./labse_indic_finetuning")

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"

# Final model used for your benchmark notebook.
MODEL_OUTPUT_DIR = OUTPUT_DIR / "labse_indic_finetuned"

# Best model according to validation evaluator.
BEST_MODEL_DIR = OUTPUT_DIR / "labse_indic_finetuned_best"

# Rolling checkpoints used for crash recovery / resume.
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

LOG_DIR = OUTPUT_DIR / "logs"

for d in [BASE_DIR, DATA_DIR, OUTPUT_DIR, MODEL_OUTPUT_DIR, BEST_MODEL_DIR, CHECKPOINT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Base folder:", BASE_DIR)
print("Data folder:", DATA_DIR)
print("Final model folder:", MODEL_OUTPUT_DIR)
print("Best model folder:", BEST_MODEL_DIR)
print("Checkpoint folder:", CHECKPOINT_DIR)


## 4. Training configuration

This version is configured for **10 epochs**.

For a smoke test, keep `QUICK_TRAIN_N = 5000`.  
For full training, set `QUICK_TRAIN_N = 0`.

Checkpointing is enabled by default. The notebook saves:

- rolling checkpoints during training,
- the best validation model,
- the final model,
- the exact train/validation split used.


In [ ]:
BASE_MODEL_NAME = "sentence-transformers/LaBSE"

MAX_SEQ_LENGTH = 128
BATCH_SIZE = 32
EPOCHS =20
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.10

# Set to 0 for full training.
QUICK_TRAIN_N = 5000

# Optional cap for validation retrieval evaluator.
VAL_MAX_N = 1000

# If GPU is available, AMP reduces memory usage and can speed up training.
USE_AMP = torch.cuda.is_available()

# -----------------------------
# Checkpointing settings
# -----------------------------

# Resume automatically if a valid SentenceTransformer checkpoint already exists.
RESUME_FROM_LATEST_CHECKPOINT = True

# Save checkpoints twice per epoch. This is safer than only saving at the end of each epoch.
# For a very unstable runtime, you can increase this to 4.
CHECKPOINTS_PER_EPOCH = 2

# Never checkpoint/evaluate less often than this, unless the epoch itself has fewer steps.
MIN_CHECKPOINT_STEPS = 100

# Keep the most recent 20 rolling checkpoints.
# For 10 epochs and 2 checkpoints per epoch, this keeps the full rolling history.
CHECKPOINT_SAVE_TOTAL_LIMIT = 20

print("Base model:", BASE_MODEL_NAME)
print("Max sequence length:", MAX_SEQ_LENGTH)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Quick train N:", QUICK_TRAIN_N)
print("Use AMP:", USE_AMP)
print("Resume from latest checkpoint:", RESUME_FROM_LATEST_CHECKPOINT)
print("Checkpoints per epoch:", CHECKPOINTS_PER_EPOCH)
print("Checkpoint save total limit:", CHECKPOINT_SAVE_TOTAL_LIMIT)


## 5. Load LaBSE or resume from latest checkpoint

If the runtime disconnects, rerun the setup cells.  
With `RESUME_FROM_LATEST_CHECKPOINT = True`, this cell will load the latest checkpoint from `CHECKPOINT_DIR` if one exists.


In [ ]:
def find_latest_sentence_transformer_checkpoint(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    if not checkpoint_dir.exists():
        return None

    candidates = []
    for p in checkpoint_dir.glob("*"):
        if not p.is_dir():
            continue

        # SentenceTransformer checkpoints normally contain modules.json.
        if (p / "modules.json").exists():
            candidates.append(p)

    if not candidates:
        return None

    return max(candidates, key=lambda p: p.stat().st_mtime)

latest_checkpoint = find_latest_sentence_transformer_checkpoint(CHECKPOINT_DIR)

if RESUME_FROM_LATEST_CHECKPOINT and latest_checkpoint is not None:
    MODEL_TO_LOAD = str(latest_checkpoint)
    print("Resuming from latest checkpoint:", MODEL_TO_LOAD)
else:
    MODEL_TO_LOAD = BASE_MODEL_NAME
    print("Loading base model:", MODEL_TO_LOAD)

model = SentenceTransformer(MODEL_TO_LOAD, device=DEVICE)
model.max_seq_length = MAX_SEQ_LENGTH

print(model)
print("Max sequence length:", model.max_seq_length)


## 6. DATASET LOADING BLANK SPOT

Paste your Samanantar / BPCC loading code in the cell below.

By the end of the cell, you must create:

```python
train_df
```

with exactly these required columns:

```text
sentence1
sentence2
```

Example:

| sentence1 | sentence2 |
|---|---|
| Hindi sentence | Tamil sentence |
| Telugu sentence | Bengali sentence |
| English sentence | Hindi sentence |

For alignment-focused fine-tuning, you can use:
- English ↔ Indic pairs
- Indic ↔ Indic pairs created through an English pivot
- A mixture of both

In [ ]:
# ============================================================
# TODO: LOAD YOUR DATASET HERE
# ============================================================

# Your final dataframe must be named: train_df
# Required columns:
#   train_df["sentence1"]
#   train_df["sentence2"]

# Example only. DELETE this and replace with real Samanantar/BPCC loading.
# train_df = pd.DataFrame({
#     "sentence1": [
#         "यह एक उदाहरण वाक्य है।",
#         "இது ஒரு உதாரண வாக்கியம்.",
#     ],
#     "sentence2": [
#         "This is an example sentence.",
#         "This is an example sentence.",
#     ],
# })

# ============================================================
# END DATASET LOADING
# ============================================================

## 7. Dataset sanity check and cleaning

In [ ]:
if "train_df" not in globals():
    raise ValueError(
        "train_df is not defined. Go back to the dataset-loading cell and create train_df "
        "with columns: sentence1, sentence2"
    )

required_cols = ["sentence1", "sentence2"]

for col in required_cols:
    if col not in train_df.columns:
        raise ValueError(f"Missing required column: {col}")

train_df = train_df[required_cols].copy()

train_df["sentence1"] = train_df["sentence1"].astype(str).str.strip()
train_df["sentence2"] = train_df["sentence2"].astype(str).str.strip()

train_df = train_df[
    (train_df["sentence1"] != "") &
    (train_df["sentence2"] != "") &
    (train_df["sentence1"].str.lower() != "nan") &
    (train_df["sentence2"].str.lower() != "nan")
].copy()

train_df = train_df.drop_duplicates(subset=["sentence1", "sentence2"]).reset_index(drop=True)

if QUICK_TRAIN_N and QUICK_TRAIN_N > 0:
    train_df = train_df.sample(
        n=min(QUICK_TRAIN_N, len(train_df)),
        random_state=SEED
    ).reset_index(drop=True)

print("Training pairs after cleaning:", len(train_df))
display(train_df.head())

## 8. Train-validation split

The validation split here is only a training-time sanity check.  
Your real evaluation should still be done using your IN22 benchmark notebook.

In [ ]:
if len(train_df) < 100:
    raise ValueError("Training data is too small. Add more pairs before fine-tuning.")

train_pairs, val_pairs = train_test_split(
    train_df,
    test_size=0.02,
    random_state=SEED,
    shuffle=True
)

train_pairs = train_pairs.reset_index(drop=True)
val_pairs = val_pairs.reset_index(drop=True)

if VAL_MAX_N and len(val_pairs) > VAL_MAX_N:
    val_pairs = val_pairs.sample(n=VAL_MAX_N, random_state=SEED).reset_index(drop=True)

# Save the exact split used so the run is reproducible and recoverable.
train_pairs_path = DATA_DIR / "train_pairs_used.csv"
val_pairs_path = DATA_DIR / "val_pairs_used.csv"
train_pairs.to_csv(train_pairs_path, index=False)
val_pairs.to_csv(val_pairs_path, index=False)

print("Train pairs:", len(train_pairs))
print("Validation pairs:", len(val_pairs))
print("Saved train split to:", train_pairs_path)
print("Saved validation split to:", val_pairs_path)


## 9. Convert pairs to SentenceTransformers format

`MultipleNegativesRankingLoss` uses in-batch negatives.

For a batch like:

```text
Hindi_1 ↔ Tamil_1
Hindi_2 ↔ Tamil_2
Hindi_3 ↔ Tamil_3
```

For `Hindi_1`, only `Tamil_1` is the positive.  
`Tamil_2`, `Tamil_3`, etc. become negatives automatically.

In [ ]:
train_examples = [
    InputExample(texts=[row["sentence1"], row["sentence2"]])
    for _, row in train_pairs.iterrows()
]

train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=BATCH_SIZE,
    drop_last=True
)

train_loss = losses.MultipleNegativesRankingLoss(model)

warmup_steps = math.ceil(len(train_dataloader) * EPOCHS * WARMUP_RATIO)

print("Train examples:", len(train_examples))
print("Train batches:", len(train_dataloader))
print("Warmup steps:", warmup_steps)

## 10. Validation evaluator

This uses aligned validation pairs and checks whether each source sentence retrieves its paired target sentence.  
It is not a replacement for the full IN22 benchmark.

In [ ]:
if len(val_pairs) > 0:
    evaluator = TranslationEvaluator(
        source_sentences=val_pairs["sentence1"].tolist(),
        target_sentences=val_pairs["sentence2"].tolist(),
        name="val_translation_retrieval",
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
    )
else:
    evaluator = None

print("Evaluator ready:", evaluator is not None)

## 11. Fine-tune LaBSE with 10 epochs and checkpointing

This cell saves:

- **rolling checkpoints** in `CHECKPOINT_DIR`,
- **best validation model** in `BEST_MODEL_DIR`,
- training configuration in `training_config.json`.

The final model is saved explicitly in the next cell.


In [ ]:
steps_per_epoch = len(train_dataloader)

if steps_per_epoch == 0:
    raise ValueError("No training batches found. Check BATCH_SIZE and training data size.")

# Save/evaluate at regular points inside every epoch.
checkpoint_save_steps = max(1, steps_per_epoch // CHECKPOINTS_PER_EPOCH)
checkpoint_save_steps = max(MIN_CHECKPOINT_STEPS, checkpoint_save_steps)
checkpoint_save_steps = min(checkpoint_save_steps, steps_per_epoch)

evaluation_steps = checkpoint_save_steps

training_config = {
    "base_model_name": BASE_MODEL_NAME,
    "loaded_model": MODEL_TO_LOAD,
    "max_seq_length": MAX_SEQ_LENGTH,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "warmup_steps": warmup_steps,
    "steps_per_epoch": steps_per_epoch,
    "evaluation_steps": evaluation_steps,
    "checkpoint_save_steps": checkpoint_save_steps,
    "checkpoint_save_total_limit": CHECKPOINT_SAVE_TOTAL_LIMIT,
    "quick_train_n": QUICK_TRAIN_N,
    "train_pairs": len(train_pairs),
    "validation_pairs": len(val_pairs),
    "use_amp": USE_AMP,
    "final_model_dir": str(MODEL_OUTPUT_DIR),
    "best_model_dir": str(BEST_MODEL_DIR),
    "checkpoint_dir": str(CHECKPOINT_DIR),
}

config_path = OUTPUT_DIR / "training_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(training_config, f, indent=2, ensure_ascii=False)

print("Steps per epoch:", steps_per_epoch)
print("Evaluation steps:", evaluation_steps)
print("Checkpoint save steps:", checkpoint_save_steps)
print("Checkpoint folder:", CHECKPOINT_DIR)
print("Best model folder:", BEST_MODEL_DIR)
print("Training config saved to:", config_path)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    output_path=str(BEST_MODEL_DIR),
    optimizer_params={"lr": LEARNING_RATE},
    evaluation_steps=evaluation_steps,
    save_best_model=True,
    show_progress_bar=True,
    use_amp=USE_AMP,
    checkpoint_path=str(CHECKPOINT_DIR),
    checkpoint_save_steps=checkpoint_save_steps,
    checkpoint_save_total_limit=CHECKPOINT_SAVE_TOTAL_LIMIT,
)


## 12. Save final model explicitly

`BEST_MODEL_DIR` contains the best validation model saved during training.  
`MODEL_OUTPUT_DIR` contains the final model after epoch 10.  
Usually you should benchmark both once, then keep whichever gives better IN22 results.


In [ ]:
model.save(str(MODEL_OUTPUT_DIR))

print("Saved final fine-tuned model to:")
print(MODEL_OUTPUT_DIR)
print("Best validation model is at:")
print(BEST_MODEL_DIR)
print("Rolling checkpoints are at:")
print(CHECKPOINT_DIR)


## 13. Quick load test

This tests the **final epoch model** from `MODEL_OUTPUT_DIR`.  
You can switch to `BEST_MODEL_DIR` below if you want to test the best validation model instead.


In [ ]:
del model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Change this to BEST_MODEL_DIR if you want to test the best validation model instead.
MODEL_FOR_QUICK_TEST = MODEL_OUTPUT_DIR

finetuned_model = SentenceTransformer(str(MODEL_FOR_QUICK_TEST), device=DEVICE)
finetuned_model.max_seq_length = MAX_SEQ_LENGTH

test_sentences = [
    "यह एक परीक्षण वाक्य है।",
    "இது ஒரு சோதனை வாக்கியம்.",
    "This is a test sentence."
]

emb = finetuned_model.encode(
    test_sentences,
    normalize_embeddings=True,
    convert_to_numpy=True,
    batch_size=BATCH_SIZE,
    show_progress_bar=False,
)

print("Loaded model from:", MODEL_FOR_QUICK_TEST)
print("Embedding shape:", emb.shape)
print("Cosine Hindi-Tamil:", float(np.dot(emb[0], emb[1])))
print("Cosine Hindi-English:", float(np.dot(emb[0], emb[2])))


## 14. Add the fine-tuned model to your benchmark notebook

After training, benchmark both the final model and best-validation model once.

For Colab Drive path:

```python
{
    'name': 'labse_indic_finetuned_final',
    'hf_id': '/content/drive/MyDrive/labse_indic_finetuning/outputs/labse_indic_finetuned',
    'kind': 'sentence_transformer'
},
{
    'name': 'labse_indic_finetuned_best',
    'hf_id': '/content/drive/MyDrive/labse_indic_finetuning/outputs/labse_indic_finetuned_best',
    'kind': 'sentence_transformer'
}
```

For Kaggle output path:

```python
{
    'name': 'labse_indic_finetuned_final',
    'hf_id': '/kaggle/working/labse_indic_finetuning/outputs/labse_indic_finetuned',
    'kind': 'sentence_transformer'
},
{
    'name': 'labse_indic_finetuned_best',
    'hf_id': '/kaggle/working/labse_indic_finetuning/outputs/labse_indic_finetuned_best',
    'kind': 'sentence_transformer'
}
```

Then benchmark only a small comparison set first:

```python
RUN_MODEL_NAMES = [
    'labse',
    'labse_indic_finetuned_final',
    'labse_indic_finetuned_best',
    'vyakyarth',
    'bge_m3',
    'multilingual_e5_large_instruct'
]
```

Success means the fine-tuned model improves or preserves:

- cosine gap
- specificity at threshold 0.80
- balanced accuracy
- ROC-AUC
- Accuracy@1 / MRR

Watch carefully that random-pair cosine does not increase too much.
